### **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import faiss

c:\Users\Micro Tech\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Load Dataset**

I use the Quora Question Pairs dataset, which contains real user questions.

In [2]:
from datasets import load_dataset

dataset = load_dataset("AlekseyKorshuk/quora-question-pairs")
data = dataset["train"]

In [3]:
print(data)

Dataset({
    features: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'],
    num_rows: 404290
})


In [4]:
quora_df = pd.DataFrame(data)
quora_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


So, for semantic search, We do not need:
- qid1
- qid2
- id
- is_duplicate

we only need the text (question1 and question2).

For this, we combine both question columns into one searchable corpus. Because if we only use question1, we lose half of the dataset.

### **Data Preprocessing**

#### **Handling Missing Values**

In [5]:
quora_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64

In [6]:
quora_df = quora_df.dropna(subset=['question1', 'question2'])

In [7]:
# Sanity check
quora_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

#### **Removing Duplicates**

In [8]:
quora_df.duplicated().sum()

np.int64(0)

#### **Corpus Creation - Merged question1 and question2**

In [9]:
corpus = pd.concat([
    quora_df['question1'],
    quora_df['question2']
], ignore_index=True) # reset numbering after merging data

In [10]:
print("Total Questions:", len(corpus))

Total Questions: 808574


In [11]:
corpus.head()

0    What is the step by step guide to invest in sh...
1    What is the story of Kohinoor (Koh-i-Noor) Dia...
2    How can I increase the speed of my internet co...
3    Why am I mentally very lonely? How can I solve...
4    Which one dissolve in water quikly sugar, salt...
dtype: str

#### **Remove Duplicate Questions**

In [12]:
duplicates_after_merge = corpus.duplicated().sum()
print("Total number of duplicated afer merging question1 and question2 :", duplicates_after_merge)

Total number of duplicated afer merging question1 and question2 : 271215


In [13]:
corpus = corpus.drop_duplicates()

In [14]:
corpus = corpus.reset_index(drop=True)

In [15]:
print("After removing duplicates:", corpus.duplicated().sum())
print("Final corpus size:", len(corpus))

After removing duplicates: 0
Final corpus size: 537359


### **Data Sampling and Text Preprocessing**

#### **Data Sampling**

In [16]:
corpus = corpus.sample(10000, random_state=42)

#### **Text Normalization**

In [17]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text) # remove punctuations
    text = re.sub(r"\s+", " ", text) # remove extra spaces
    return text.strip()

In [18]:
corpus = corpus.apply(clean_text)

### **Load Pretrained Embedding Model**

We use Sentence-BERT (MiniLM) which converts sentences into vectors.

In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Micro Tech\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Micro Tech\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6060.37it

### **Generate Embeddings**

In [20]:
texts = corpus.tolist()

corpus_embeddings = model.encode(texts,
                          batch_size=64,
                          show_progress_bar=True)

print("Embeddings shape:", corpus_embeddings.shape)

Batches: 100%|██████████| 157/157 [00:29<00:00,  5.39it/s]


Embeddings shape: (10000, 384)


### **Store Embedding in FAISS**

The embeddings were converted to float32 format because FAISS is optimized for 32-bit floating point vectors, providing better memory efficiency and faster similarity search.

In [21]:
# Convert embeddings to float32
embedding_matrix = np.array(corpus_embeddings).astype('float32')

# Get embedding dimension
dimension = embedding_matrix.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Store embeddings
index.add(embedding_matrix)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 10000


### **Semantic Search**

In [22]:
def semantic_search_cosine(query, top_k=5):

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Compute cosine similarity
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Get top-k most similar indices
    top_k_indices = similarities.argsort()[-top_k:][::-1]

    # Collect results
    results = []
    for idx in top_k_indices:
         results.append((corpus.iloc[idx], similarities[idx]))

    return results

### **Testing with 10 Queries**

In [23]:
test_queries = [
    "How can I learn programming?",
    "What is machine learning?",
    "Best way to get a job in tech?",
    "How to improve English?",
    "What is AI?",
    "How does deep learning work?",
    "How to lose weight fast?",
    "Best career options in IT?",
    "How to study effectively?",
    "What is data science?"
]

### **Top-K Retrieved Results**

In [24]:
for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"QUERY {i}: {query}")
    print(f"{'-'*80}")

    results = semantic_search_cosine(query, top_k=5)

    for j, (text, score) in enumerate(results, 1):
        print(f"{j}) {text}")
        print(f"-  Similarity Score: {score:.4f}\n")


QUERY 1: How can I learn programming?
--------------------------------------------------------------------------------
1) how do you practice programming
-  Similarity Score: 0.8271

2) what is the best method to learn coding
-  Similarity Score: 0.7086

3) how do i become great programmer
-  Similarity Score: 0.6760

4) what are the good books i have to read to learn coding from the basics
-  Similarity Score: 0.6578

5) i want to make a 3d game i know designing what are the best ways to learn game programming
-  Similarity Score: 0.6506


QUERY 2: What is machine learning?
--------------------------------------------------------------------------------
1) what is machine learning in laymans terms
-  Similarity Score: 0.9418

2) whats the difference between deep learning and machine learning
-  Similarity Score: 0.6514

3) what is deep learning
-  Similarity Score: 0.6026

4) what is a good tutorial on machine learning on youtube
-  Similarity Score: 0.5630

5) i want to have a stron